# 2D-to-3DGS — Colab orchestrator

Training code stays in `.py` modules. This notebook mounts Drive, installs deps, stages data to fast local disk (`/content/data`), and runs `train.py` via CLI.

**Setup:** `setup.sh` skips `mamba-ssm` by default (Colab often fails building those CUDA wheels). The ViM encoder uses a built-in mixer fallback. To try real Mamba: `INSTALL_MAMBA=1 bash setup.sh` (may still fail).

**Dataset:** Edit `configs/colab_config.yaml` and set **one** of `dataset_gdrive_id`, `dataset_download_url`, `dataset_drive_zip`, or `dataset_archive`, then run the download/unzip cell.

**3D cars (open license):** We cannot redistribute ShapeNet for you. To pull **many vehicle GLBs** yourself, use [Objaverse](https://huggingface.co/datasets/allenai/objaverse) (ODC-By + per-model CC licenses): after `%cd` into the repo, run `pip install -r requirements_data.txt` then e.g. `python scripts/download_objaverse_vehicles.py --out_dir /content/data/objaverse_vehicles --max_objects 500 --mode both --objaverse_cache /content/data/.objaverse_cache`. You still must **render** multi-view PNGs and add `manifest.jsonl` for this repo’s dataloader (see `data/dataset.py`).

**Imports:** Run the clone/`%cd` cell before any cell that imports `utils` or runs `train.py`. The download and train cells force `PROJECT_ROOT` on `sys.path` so rerunning a single cell still works if the repo is at `/content/2d-to-3d`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone on first run; on later runs `git pull` picks up new files (e.g. utils/dataset_download.py).
!if [ ! -d /content/2d-to-3d ]; then git clone https://github.com/ns-1456/2d-to-3d.git /content/2d-to-3d; else git -C /content/2d-to-3d pull; fi
%cd /content/2d-to-3d

In [ ]:
# Default install (no mamba-ssm CUDA build). For optional Mamba: INSTALL_MAMBA=1 bash setup.sh
!chmod +x setup.sh && bash setup.sh

In [ ]:
# Download / stage dataset to fast local NVMe (`data_root`). See configs/colab_config.yaml:
#   dataset_gdrive_id, dataset_download_url, dataset_drive_zip, or dataset_archive
!pip install -q gdown

import os
import sys

REPO_DIR = "/content/2d-to-3d"
if not os.path.isdir(os.path.join(REPO_DIR, "utils")):
    raise FileNotFoundError(
        f"Missing {REPO_DIR}/utils — run the clone + %%cd cell above first."
    )
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import yaml

from utils.colab_setup import unzip_to_local
from utils.dataset_download import resolve_local_zip_path

with open("configs/colab_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

root = cfg["data_root"]
zip_path = resolve_local_zip_path(cfg)
if zip_path:
    print("Local zip:", zip_path)
    unzip_to_local(zip_path, root, overwrite=False)
    print("Extracted to", root, "- set synthetic: false and add manifest.jsonl for real training.")
else:
    print(
        "No download source configured (gdrive id / URL / Drive zip / archive path). "
        "Training uses synthetic: true until you add data + set synthetic: false."
    )

In [ ]:
# Must run from repo root so `data` / `models` package imports resolve.
%cd /content/2d-to-3d
!python train.py --config configs/colab_config.yaml
# Resume after disconnect:
# !python train.py --config configs/colab_config.yaml --resume_from /content/drive/MyDrive/3DGS_Checkpoints